In [14]:
import pandas as pd
import numpy as np
from scipy.stats import t
import warnings
#%%
class panel_regression:
    
    def __init__(self,filename="eurodata_sq_panelc7x.csv",years="1993-2019",
                 y_var="ln_gdppc.d1",x_vars=['tas','tas2','prec','prec2'],make_vars=True,
                 make_lags=True,lag_vars=[],lag_n=3,
                 sel_kreise=False,sel_sector='',sel_grp=0):
        self.filename=filename
        self.mainpath="C://Users/nb/Desktop/paper_one/"
        self.maindf=pd.read_csv(f"{self.mainpath}{self.filename}")
        print(f'Shape of original dataframe = {self.maindf.shape}')
        if sel_kreise:
            print(f'Selecting counties based on {sel_sector} and group {sel_grp}')
            self.maindf=self.maindf[self.maindf[f'qtl_in_{sel_sector}']==sel_grp]
            self.maindf=self.maindf.drop(['share_BE', 'share_GI', 'share_A',
                                          'qtl_in_A', 'qtl_in_BE', 'share_F', 
                                          'qtl_in_F', 'qtl_in_GI'],axis=1)
            print(f'Shape of the dataframe = {self.maindf.shape}')
            self.kreise_list=list(self.maindf.Kreise_Code.unique())
            print(f'drop kreise {self.kreise_list[4]}')
        else:
            self.kreise_list=list(self.maindf.Kreise_Code.unique())
            print('Continuing with fulldf')
        self.maindf=self.maindf.sort_values(by=['Kreise_Code','year'])
        if make_vars: #'sq05_colddays_f1','sq95_hotdays_f1','sq05_dryweeks_f1','sq95_wetweeks_f1','sq_mean_precpd','sq_std_precpd','sq_std_tas','tas','prec'
            self.maindf['tas.d1']=self.maindf[['Kreise_Code','tas']].groupby('Kreise_Code').transform('diff')
            self.maindf['tasXtas.d1']=self.maindf['tas']*self.maindf['tas.d1']
            self.maindf['prec.d1']=self.maindf[['Kreise_Code','prec']].groupby('Kreise_Code').transform('diff')
            self.maindf['precXprec.d1']=self.maindf['prec']*self.maindf['prec.d1']
            self.maindf['full_hotXdry_f1']=self.maindf['full95_hotdays_f1']*self.maindf['full05_dryweeks_f1']
            self.maindf['full_hotXdry_f2']=self.maindf['full95_hotdays_f2']*self.maindf['full05_dryweeks_f2']
            self.maindf['sq_hotXdry_f1']=self.maindf['sq95_hotdays_f1']*self.maindf['sq05_dryweeks_f1']
            self.maindf['sq_hotXdry_f2']=self.maindf['sq95_hotdays_f2']*self.maindf['sq05_dryweeks_f2']
            self.maindf['tasXsq_std_tas']=self.maindf['tas']*self.maindf['sq_std_tas']
            self.maindf['tasXhot95_f1']=self.maindf['tas']*self.maindf['sq95_hotdays_f1']
            self.maindf['tasXcold05_f1']=self.maindf['tas']*self.maindf['sq05_colddays_f1']
            self.maindf['tasXdry05_f1']=self.maindf['tas']*self.maindf['sq05_dryweeks_f1']
            self.maindf['tasXwet95_f1']=self.maindf['tas']*self.maindf['sq95_wetweeks_f1']
            self.maindf['tasXhot95_f2']=self.maindf['tas']*self.maindf['sq95_hotdays_f2']
            self.maindf['tasXcold05_f2']=self.maindf['tas']*self.maindf['sq05_colddays_f2']
            self.maindf['tasXdry05_f2']=self.maindf['tas']*self.maindf['sq05_dryweeks_f2']
            self.maindf['tasXwet95_f2']=self.maindf['tas']*self.maindf['sq95_wetweeks_f2']
            
        self.maindf['year']=self.maindf['year'].astype(int)
        self.maindf['Kreise_Code']=self.maindf['Kreise_Code'].astype(int)
        self.x_vars=x_vars
        self.y_var=y_var
        self.regdf=self.maindf[['Kreise_Code','year']+[self.y_var]+self.x_vars].copy()
        print(f"Shape of the dataframe = {self.regdf.shape}")
        
        if make_lags:
            print(f'Making {lag_n} Lags: ...')
            for v in lag_vars:
                for n in range(lag_n):
                    n=n+1
                    self.regdf.loc[:,f'{v}.lag{n}']=self.regdf.groupby('Kreise_Code')[v].shift(n)
                    self.x_vars.append(f'{v}.lag{n}')
            self.regdf=self.regdf.dropna()
            print(f'First year = {self.regdf.year.min()}')
            print(f'Last year = {self.regdf.year.max()}')
        else:
            self.regdf=self.regdf.dropna()
            
        self.regdf=self.regdf[(self.regdf['year']>=int(years.split("-")[0]))&
                                (self.regdf['year']<=int(years.split("-")[1]))]    
        self.regdf=self.regdf.dropna()
        self.years_list=list(self.regdf.year.unique())
        index = pd.MultiIndex.from_product([self.kreise_list, self.years_list], names=['Kreise_Code', 'year'])
        print('self.regdf.shape = ',self.regdf.shape)
        self.trend_cols=[]
        self.dummy_cols=[]
        self.dummy_df=pd.DataFrame(index=index)
        self.trends_df=pd.DataFrame(index=index)
        nyears=self.regdf.year.max() - self.regdf.year.min() +1
        print(f'Time Frame = {self.regdf.year.min()} - {self.regdf.year.max()}')
        print(f'Number of years = {nyears}')
        print(f'Number of counties = {self.regdf.Kreise_Code.nunique()}')
        nobs=nyears*self.regdf.Kreise_Code.nunique()
        self.counties=self.regdf.Kreise_Code.nunique()
        match=(nobs==self.regdf.shape[0])
        print('Is the panel balanced ? = ',match)
        
    def _create_fixed_effect_dummies(self,absorb_fe="both",
                                 drop_unit=3257, drop_time=2012):
        drop_unit=int(drop_unit)
        drop_time=int(drop_time)
        drop_kreise=f"dummy_kr{drop_unit}"
        drop_year=f"dummy_yr{drop_time}"
        self.dummy_list=[]
        self.dummiesdf=pd.DataFrame()
        for kreise in self.kreise_list:
            temp=pd.DataFrame()
            for year in self.years_list:
                if absorb_fe=="both":
                    temp.loc[year,f"dummy_kr{int(kreise)}"]=1
                    temp.loc[year,f"dummy_yr{int(year)}"]=1
                    drop_cols=[drop_kreise,drop_year]
                elif absorb_fe=="time":
                    temp.loc[year,f"dummy_yr{int(year)}"]=1
                    drop_cols=[drop_year]
                elif absorb_fe=="unit":
                    temp.loc[year,f"dummy_kr{int(kreise)}"]=1
                    drop_cols=[drop_kreise]
            temp=temp.reset_index().rename(columns={'index':'year'})
            temp['Kreise_Code']=kreise
            self.dummy_list.append(temp)
        
        self.dummy_df=pd.concat(self.dummy_list,axis=0)
        self.dummy_df=self.dummy_df.drop(drop_cols,axis=1)
        self.dummy_df=self.dummy_df.reset_index(drop=True).fillna(0)
        
        self.dummy_cols=[i for i in self.dummy_df.columns if 'dummy' in i]
        dummy_len=len(self.dummy_cols)
        print(f"Created {dummy_len} dummy columns.") ## should be 423+2 columns in total
        self.dummy_df = self.dummy_df.set_index(['Kreise_Code', 'year'])
        
    def _create_county_trends(self, drop_unit=3257, add_quad=True, add_cube=False):
        base = self.regdf[['Kreise_Code', 'year']].copy()
    
        # Time variables
        t = base['year'] - base['year'].min() + 1
        t2 = t ** 2
        t3 = t ** 3
        self.trend_cols = {}
    
        for kr in base['Kreise_Code'].unique():
            if kr == drop_unit:
                continue
            col_t = f"{kr}_t"
            self.trend_cols[col_t] = (base['Kreise_Code'] == kr).astype(int) * t
            if add_quad:
                col_t2 = f"{kr}_t2"
                self.trend_cols[col_t2] = (base['Kreise_Code'] == kr).astype(int) * t2
            if add_cube:
                col_t3 = f"{kr}_t3"
                self.trend_cols[col_t3] = (base['Kreise_Code'] == kr).astype(int) * t3
            #self.trend_cols.extend([col_t, col_t2])

        self.trends_df = pd.concat([base, pd.DataFrame(self.trend_cols)], axis=1)
        
        self.trend_cols=list(self.trend_cols.keys())
        print(f"Created {len(self.trend_cols)} county time trend columns.")
        self.trends_df = self.trends_df.set_index(['Kreise_Code', 'year'])
        
    def _create_custom_trends(self,trend_cols):
        self.trend_cols=trend_cols
        self.trends_df=self.maindf[['Kreise_Code','year']+trend_cols].set_index(['Kreise_Code','year'])
        
    def _projection_matrix(self):
        self.y1,self.y2=self.regdf.year.min(),self.regdf.year.max()
        
        print(f'The years of observations = {int(self.y1)}-{int(self.y2)}')
        print(f'The total number of counties = {int(self.regdf.Kreise_Code.nunique())}')
        print(f'The total number of obervations = {int(self.regdf.shape[0])}')
        self.years_list=list(range(self.y1,self.y2+1))
        self.regdf = self.regdf.set_index(['Kreise_Code', 'year'])
        print(f'The number of regression variables = {self.regdf.shape[1]}')
        print(f'The number of trends = {self.trends_df.shape[1]}')
        print(f'The number of dummies = {self.dummy_df.shape[1]}')
    
        self.regdf = (
            self.regdf
            .join(self.trends_df, how="left")
            .join(self.dummy_df, how="left")
        )

        D_cols = self.trend_cols + self.dummy_cols
        y_cols = [self.y_var]
        const_col = np.ones((len(self.regdf), 1))
        
        self.D_mat = np.column_stack([self.regdf[col].to_numpy() for col in D_cols])
        temp = np.linalg.inv(self.D_mat.T @ self.D_mat)
        print("Checking if the D matrix is full rank ...")
        self.d_rank = np.linalg.matrix_rank(self.D_mat)
        self.N=self.regdf.shape[0]
        self.freedom=self.N-len(self.x_vars)-self.d_rank
        if self.d_rank == self.D_mat.shape[1]:
            print('D matrix is at full rank. Proceed. uWu.')
        else:
           warnings.warn("D matrix is collinear: Projection may be invalid. uWu.", UserWarning)
                    
        P_D = self.D_mat @ temp @ self.D_mat.T
        self.M_D = np.eye(P_D.shape[0]) - P_D
        
        self.X_mat = np.hstack([const_col, self.regdf[self.x_vars].to_numpy()])
        self.y_mat = self.regdf[y_cols].to_numpy()

        print(f'Shape of D Matrix = {self.D_mat.shape}')
        print(f'Shape of X Matrix = {self.X_mat.shape}')
    
    def _do_OLS(self):
        
        self.X_tilde = self.M_D @ self.X_mat
        self.y_tilde = self.M_D @ self.y_mat
        
        self.beta_tilde = np.linalg.inv(self.X_tilde.T @ self.X_tilde) @ (self.X_tilde.T @ self.y_tilde)
        
        self.coef_n=self.beta_tilde.shape[0]
        print(f"Estimated {self.coef_n} Coefficients from Regression.")
        self.y_hat_tilde=self.X_tilde @ self.beta_tilde
        
        self.u_tilde = self.y_tilde - self.y_hat_tilde
        
    def _do_anova(self,cluster="kreise"):
        self.cluster_sum = np.zeros((self.coef_n, self.coef_n))
        if cluster=="kreise":
            self.clusters=self.regdf.index.get_level_values('Kreise_Code')
            print("Standard Errors are clustered at the county-level")
            for kreise in self.kreise_list:
                kreise_mask = kreise == self.clusters
                x_cluster = self.X_tilde[kreise_mask,:] 
                u_cluster = self.u_tilde[kreise_mask,:]
                self.cluster_sum = self.cluster_sum + (x_cluster.T @ u_cluster)@((x_cluster.T @ u_cluster).T)
        elif cluster=="year":
            self.clusters=self.regdf.index.get_level_values('year')
            print("Standard Errors are clustered at the year-level")
            for year in self.years_list:
                year_mask = year == self.clusters
                x_cluster = self.X_tilde[year_mask,:] 
                u_cluster = self.u_tilde[year_mask,:]
                self.cluster_sum = self.cluster_sum + (x_cluster.T @ u_cluster)@((x_cluster.T @ u_cluster).T)
        elif cluster=="conley":
            self.conley_wts=pd.read_csv("C:\\Users/nb/Desktop/paper_one/conley_weights.csv")
            self.conley_wts=self.conley_wts.set_index('Unnamed: 0')
            print("Standard Errors are Conley Standard Errors with bound of 250km")
            all_obs=self.regdf.index
            all_xu={}
            for obs,(kreise,year) in enumerate(all_obs):
                xu_tilde=self.X_tilde[obs,:].reshape(-1,1) @ self.u_tilde[obs].reshape(1,1)
                all_xu[(kreise,year)]=xu_tilde
            years=list(self.regdf.index.get_level_values('year'))
            for item in all_xu:
                outer=all_xu[item]
                k1=item[0]
                temp_xu=list(self.conley_wts[(self.conley_wts[str(k1)]>0)].index)
                pos_xu = [(county, year) for county, year in all_xu.keys() if county in temp_xu]
                for jtem in pos_xu:
                    k2=jtem[0]
                    w_ij=self.conley_wts.loc[k1,str(k2)]
                    self.cluster_sum=self.cluster_sum+(w_ij * outer@all_xu[jtem].T)
        else:
            print("Cluster can only take kreise, year or conley as inputs")
        se_denominator = np.linalg.inv(self.X_tilde.T @ self.X_tilde)
        self.beta_cov = se_denominator @ self.cluster_sum @ se_denominator
        self.beta_se = np.sqrt(np.diag(self.beta_cov)).reshape(self.coef_n,1)
        
        self.t_stats = self.beta_tilde/self.beta_se
        print(f'The degrees of freedom = {self.freedom}')
        self.p_values = 2 * t.sf(np.abs(self.t_stats), self.freedom)
        
        self.y_fe = np.round(self.y_mat - self.y_tilde,6)
        self.SSR=np.sum(self.u_tilde**2)
        self.SST_tilde=np.sum(self.y_tilde**2)
        self.within_r2 = 1- (self.SSR / self.SST_tilde)
        self.SST=np.sum((self.y_mat - np.mean(self.y_mat))**2)
        temp=(self.N-1)/ self.freedom
        self.adjusted_r2 = 1- (temp*self.SSR/self.SST)

        
    def _model_summary(self):
        print("uWu ************* RESULTS SUMMARY ***************** uWu")
        print("=======================================================")
        print(f"The dependent variable = {self.y_var}")
        print("-------------------------------------------------------")
        print(f'Within R2 = {np.round(self.within_r2,6)}')
        print(f'Adjusted R2 = {np.round(self.adjusted_r2,6)}')
        print("-------------------------------------------------------")
        
        self.summary_df=pd.DataFrame(index=["const"]+self.x_vars)
        self.summary_df["coefs"]=np.round(self.beta_tilde,5)
        self.summary_df["std er"]=np.round(self.beta_se,4)
        self.summary_df["t-stat"]=np.round(self.t_stats,4)
        self.summary_df["p_values"]=np.round(self.p_values,2)
        print(self.summary_df.sort_index())
        print("*******************************************************")


In [4]:
def make_coef_df(indf, model_id,keep_hist=True,dist_lag=True):
    indf=indf.reset_index()
    indf['var_name']=indf['index'].apply(lambda x: x.split('.')[0])
    vars_f1=[f'sq05_{i}_f1' for i in ['colddays','dryweeks'] ]+[f'sq95_{i}_f1' for i in ['hotdays','wetweeks']]
    vars_f2=[f'sq05_{i}_f2' for i in ['colddays','dryweeks'] ]+[f'sq95_{i}_f2' for i in ['hotdays','wetweeks']]
    vars_f1=vars_f1+['sq_mean_precpd','sq_std_precpd']
    vars_f2=vars_f2+['sq_mean_precpd','sq_std_precpd']
    if dist_lag:
        indf['star']=1
    else:
        indf['star'] = (indf['p_values'] <= 0.10).astype(int)
    indf['coef_star']=indf['star']*indf['coefs']
    indf=indf[['var_name','coef_star']].groupby('var_name').sum()
    if 'f1' in model_id:
        indf=indf.loc[vars_f1,'coef_star']
    elif 'f2'in model_id:
        indf=indf.loc[vars_f2,'coef_star']
    indf=indf.reset_index()
    indf.columns=['var_name',model_id.split('.')[-1]]
    if keep_hist == False:
        indf.var_name=indf.var_name.apply(lambda x: x.replace('_f1','').replace('_f2',''))
    return indf.set_index('var_name')

In [5]:
def design_table(resdfs, time_trends, dummies, years, counties, whn_r2, adj_r2, history,are_lags=False, are_interact=False):

    print('\\begin{table}[ht]')
    print('  \\centering')
    print('  \\caption{Estimation Results: Impact of Weather Anomalies on Growth}')
    print('  \\label{tab:regression_results}')
    
    # Column setup: Left align first column, center the rest
    col_config = 'l' + 'c' * len(resdfs)
    print('  \\begin{tabular}{' + col_config + '}')
    print('    \\toprule')
    
    # Header Row
    header = 'Variable'
    for i in range(len(resdfs)):
        header += f' & ({i+1})'
    print(f'    {header} \\\\')
    print('    \\midrule')

    var_dict = {
        'sq05_colddays': 'Annual Cold Days',
        'sq95_hotdays': 'Annual Hot Days',
        'sq05_dryweeks': 'Annual Dry Weeks',
        'sq95_wetweeks': 'Annual Wet Weeks',
        'sq_std_tas': 'Annual Var. Temp ($^\\circ$C)',
        'sq_std_precpd': 'Annual Var. Precip (mm/day)',
        'sq_mean_precpd': 'Annual Mean Precip (mm/day)',
        'prec': 'Total Annual Precip',
        'tas': 'Mean Annual Temp ($^\\circ$C)',
        'const': 'Constant'
    }
    
    var_dict_lags=var_dict.copy()
    if are_lags:
        for item in var_dict:
            var_dict_lags[f'{item}.lag1']=var_dict[item]+'.L1'
            var_dict_lags[f'{item}.lag2']=var_dict[item]+'.L2'
    
    if are_interact:
        var_dict_lags['tasXsq_std_tas']='Temp X StDev_Temp'
        var_dict_lags['tasXhot95']='Temp X Hot Days'
        var_dict_lags['tasXcold05']='Temp X Cold Days'
        var_dict_lags['tasXdry05']='Temp X Dry Weeks'
        var_dict_lags['tasXwet95']='Temp X Wet Weeks'
    for var, label in var_dict_lags.items():
        line_coefs = f'    {label}'
        line_se = '    ' 
        for resdf in resdfs:
            resdf = resdf.reset_index()
            resdf['index'] = resdf['index'].apply(lambda x: x.replace('_f1','').replace('_f2',''))
            resdf = resdf.set_index('index')
            
            try:
                row = resdf.loc[var, :]
                val = row['coefs']
                se = row['std er']
                p = row['p_values']
                
                # Significance Stars (Academic Standard)
                stars = ''
                if p <= 0.01: stars = '$^{***}$'
                elif p <= 0.05: stars = '$^{**}$'
                elif p <= 0.1: stars = '$^{*}$'
                
                # Format to 4 decimal places
                line_coefs += f' & {val:.4f}{stars}'
                line_se += f' & ({se:.4f})'
            except KeyError:
                line_coefs += ' & '
                line_se += ' & '

        print(f'{line_coefs} \\\\')
        print(f'{line_se} \\\\')
        print('    \\addlinespace') 

    print('    \\midrule')
    
    # Summary Statistics Block
    def format_stat_row(name, stat_list):
        return f'    {name} & ' + ' & '.join(map(str, stat_list)) + ' \\\\'

    print(format_stat_row('Fixed Effects', dummies))
    print(format_stat_row('Time Trends', time_trends))
    print(format_stat_row('Years', years))
    print(format_stat_row('Counties', counties))
    print(format_stat_row('Within $R^2$', whn_r2))
    print(format_stat_row('Adj. $R^2$', adj_r2))
    print(format_stat_row('History', history))
    
    print('    \\bottomrule')
    print('    \\multicolumn{' + str(len(resdfs)+1) + '}{l}{\\small \\textit{Notes:} Standard errors in parentheses. $^{***}$ p$<$0.01, $^{**}$ p$<$0.05, $^{*}$ p$<$0.1.} \\\\')
    print('  \\end{tabular}')
    print('\\end{table}')

In [6]:
import matplotlib.pyplot as plt
def make_transpose(df):
    df.columns=[i.split('_')[-1].capitalize() for i in df.columns]
    df=df.T
    return df
def make_net_effects(c_f1, f1, c_f2, f2, plot=True, fsize=(28,18), fontsize=22, title_fontsize=26, suptitle_fontsize=30,suptitle_text=""):
    if not plot: return
    fig, axes = plt.subplots(2,2,figsize=fsize)
    c_f1.plot(ax=axes[0,0],linewidth=3); f1.plot(ax=axes[0,1],linewidth=3); c_f2.plot(ax=axes[1,0],linewidth=3); f2.plot(ax=axes[1,1],linewidth=3)
    axes[0,0].set_title("C_F1", fontsize=title_fontsize); axes[0,1].set_title("F1", fontsize=title_fontsize)
    axes[1,0].set_title("C_F2", fontsize=title_fontsize); axes[1,1].set_title("F2", fontsize=title_fontsize)
    for ax in axes.ravel():
        ax.tick_params(axis='both', labelsize=fontsize)
        ax.set_xlabel(ax.get_xlabel(), fontsize=fontsize); ax.set_ylabel(ax.get_ylabel(), fontsize=fontsize)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5,1.02), ncol=2, fontsize=fontsize, frameon=True)
    fig.suptitle(suptitle_text, fontsize=suptitle_fontsize)
    plt.tight_layout(rect=[0,0,1,0.95])
    plt.show()

# GDPPC

In [5]:
#%% gdp-percapita growth rate (f1-history), two-fe and quadratic county time trends
gdppc_f1=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=False,make_vars=False)
gdppc_f1._create_county_trends()
gdppc_f1._create_fixed_effect_dummies()
gdppc_f1._projection_matrix()
gdppc_f1._do_OLS()
gdppc_f1._do_anova()
print("Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.")
gdppc_f1._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
self.regdf.shape =  (10000, 12)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 10
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8770
Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
---------------------------------

In [70]:
tabf1_1=gdppc_f1.summary_df ##adj_r2=0.23

In [59]:
#%% gdp-percapita growth rate (f2-history), two-fe and quadratic county time trends
gdppc_f2=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f2',
                                                    'sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=False,make_vars=False)
gdppc_f2._create_county_trends()
gdppc_f2._create_fixed_effect_dummies()
gdppc_f2._projection_matrix()
gdppc_f2._do_OLS()
gdppc_f2._do_anova()
print("Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.")
gdppc_f2._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
self.regdf.shape =  (10000, 12)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 10
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8770
Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
--------------------------------

In [60]:
tabf2_1=gdppc_f2.summary_df

In [61]:
#%% gdp-percapita growth rate (f1-history), two-fe and no county time trend, clustering standard errors for counties

gdppc_f1=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=False,make_vars=False)
#gdppc_f1._create_county_trends(add_cube=False, add_quad=False)
gdppc_f1._create_fixed_effect_dummies(absorb_fe='both')
gdppc_f1._projection_matrix()
gdppc_f1._do_OLS()
gdppc_f1._do_anova()
print("Model with Seasonal Adjustments and Fixed Historical Reference Period.")
gdppc_f1._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
self.regdf.shape =  (10000, 12)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 10
The number of trends = 0
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 423)
Shape of X Matrix = (10000, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 9568
Model with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
-------------------------------------------------------
Within R2 = 0.002985
Adjusted R2 = 0.244

In [62]:
tabf1_2=gdppc_f1.summary_df

In [63]:
#%% gdp-percapita growth rate (f2-history), two-fe and no county time trend, clustering standard errors for counties

gdppc_f2=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=False,make_vars=False)
#gdppc_f2._create_county_trends(add_cube=False, add_quad=False)
gdppc_f2._create_fixed_effect_dummies(absorb_fe='both')
gdppc_f2._projection_matrix()
gdppc_f2._do_OLS()
gdppc_f2._do_anova()
print("Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.")
gdppc_f2._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
self.regdf.shape =  (10000, 12)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 10
The number of trends = 0
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 423)
Shape of X Matrix = (10000, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 9568
Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
-------------------------------------------------------
Within R2 = 0.0026

In [64]:
tabf2_2=gdppc_f2.summary_df

In [67]:
design_table(resdfs=[np.round(tabf1_1,4),np.round(tabf2_1,4),
                     np.round(tabf1_2,4),np.round(tabf2_2,4)],time_trends=['County Quadratic','County Quadratic',
                                                                   'None','None'],
             dummies=['county and year','county and year','county and year','county and year'],
             years=['1995-2019','1995-2019','1995-2019','1995-2019'],counties=['400','400','400','400'],
             whn_r2=['0.00','0.00','0.00','0.00'],
             adj_r2=['0.23','0.23','0.24','0.24'],
             history=['Fixed','Moving','Fixed','Moving'])

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lcccc}
    \toprule
    Variable & (1) & (2) & (3) & (4) \\
    \midrule
    Annual Cold Days & -0.0004$^{***}$ & -0.0004$^{***}$ & -0.0004$^{**}$ & -0.0003$^{**}$ \\
     & (0.0000) & (0.0001) & (0.0001) & (0.0001) \\
    \addlinespace
    Annual Hot Days & -0.0004$^{***}$ & -0.0004$^{***}$ & -0.0003$^{***}$ & -0.0003$^{***}$ \\
     & (0.0000) & (0.0001) & (0.0001) & (0.0001) \\
    \addlinespace
    Annual Dry Weeks & -0.0003$^{***}$ & -0.0003 & -0.0003 & -0.0003 \\
     & (0.0000) & (0.0003) & (0.0003) & (0.0003) \\
    \addlinespace
    Annual Wet Weeks & -0.0002$^{***}$ & -0.0000 & -0.0004 & -0.0003 \\
     & (0.0000) & (0.0003) & (0.0003) & (0.0003) \\
    \addlinespace
    Annual Var. Temp ($^\circ$C) & 0.0142$^{***}$ & 0.0124$^{***}$ & 0.0138$^{***}$ & 0.0120$^{***}$ \\
     & (0.0000) & (0.0030) & (0.0034) & (0.0029) \\
    \a

# Sector A 

In [5]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F1_ALL
Amod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
Amod_sq_f1._create_county_trends()
Amod_sq_f1._create_fixed_effect_dummies()
Amod_sq_f1._projection_matrix()
Amod_sq_f1._do_OLS()
Amod_sq_f1._do_anova()
print("Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.")
Amod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8371
Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_A.d1
-------------------------------------------

In [6]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F2_ALL
Amod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
Amod_sq_f2._create_county_trends()
Amod_sq_f2._create_fixed_effect_dummies()
Amod_sq_f2._projection_matrix()
Amod_sq_f2._do_OLS()
Amod_sq_f2._do_anova()
print("Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.")
Amod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8371
Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_A.d1
-----------------------------------------

# Sector BE

In [7]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_F1_ALL
BEmod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
BEmod_sq_f1._create_county_trends()
BEmod_sq_f1._create_fixed_effect_dummies()
BEmod_sq_f1._projection_matrix()
BEmod_sq_f1._do_OLS()
BEmod_sq_f1._do_anova()
print("Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.")
BEmod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8371
Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_BE.d1
-----------------------------------------

In [8]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_F2_ALL
BEmod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
BEmod_sq_f2._create_county_trends()
BEmod_sq_f2._create_fixed_effect_dummies()
BEmod_sq_f2._projection_matrix()
BEmod_sq_f2._do_OLS()
BEmod_sq_f2._do_anova()
print("Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.")
BEmod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8371
Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_BE.d1
----------------------------------------

# Sector GI

In [9]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_F1_ALL
GImod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
GImod_sq_f1._create_county_trends()
GImod_sq_f1._create_fixed_effect_dummies()
GImod_sq_f1._projection_matrix()
GImod_sq_f1._do_OLS()
GImod_sq_f1._do_anova()
print("Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.")
GImod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8371
Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_GI.d1
-----------------------------------------

In [10]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_F2_ALL
GImod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
GImod_sq_f2._create_county_trends()
GImod_sq_f2._create_fixed_effect_dummies()
GImod_sq_f2._projection_matrix()
GImod_sq_f2._do_OLS()
GImod_sq_f2._do_anova()
print("Model for Sector GI with Seasonal Adjustments and Rolling Historical Reference Period.")
GImod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8371
Model for Sector GI with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_GI.d1
---------------------------------------

In [11]:
design_table(resdfs=[Amod_sq_f1.summary_df,Amod_sq_f2.summary_df,
                     BEmod_sq_f1.summary_df,BEmod_sq_f2.summary_df,
                     GImod_sq_f1.summary_df,GImod_sq_f2.summary_df],
             time_trends=['Quad County']*6,
             dummies=['county and year']*6,
             years=['1996-2019']*6,
             counties=['400']*6,
             whn_r2=['0']*6,
             adj_r2=['0']*6,
             history=['Fixed','Moving']*3)

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lcccccc}
    \toprule
    Variable & (1) & (2) & (3) & (4) & (5) & (6) \\
    \midrule
    Annual Cold Days & -0.0034$^{***}$ & -0.0015$^{**}$ & -0.0005 & -0.0002 & -0.0002 & -0.0004$^{**}$ \\
     & (0.0007) & (0.0007) & (0.0004) & (0.0004) & (0.0002) & (0.0002) \\
    \addlinespace
    Annual Hot Days & -0.0006$^{*}$ & -0.0000 & -0.0000 & -0.0003 & 0.0000 & -0.0000 \\
     & (0.0004) & (0.0004) & (0.0003) & (0.0003) & (0.0001) & (0.0001) \\
    \addlinespace
    Annual Dry Weeks & 0.0001 & 0.0007 & -0.0001 & -0.0005 & -0.0006$^{**}$ & -0.0003 \\
     & (0.0010) & (0.0011) & (0.0008) & (0.0008) & (0.0003) & (0.0003) \\
    \addlinespace
    Annual Wet Weeks & 0.0011 & 0.0019 & -0.0019$^{**}$ & -0.0016$^{*}$ & 0.0006$^{*}$ & 0.0005 \\
     & (0.0013) & (0.0013) & (0.0010) & (0.0009) & (0.0004) & (0.0004) \\
    \addlinespace
    Annual 

# APPENDIX

## Divergence between Fixed and Rolling Distributions

In [51]:
import scipy.stats as sc

def calculate_beta_Z(df1, df2):
    df1.index = [i.replace('_f1', '').replace('_f2', '') for i in df1.index]
    df2.index = [i.replace('_f1', '').replace('_f2', '') for i in df2.index]
    
    se_diff = np.sqrt(df1['std er']**2 + df2['std er']**2)
    z_vals = (df1['coefs'] - df2['coefs']) / se_diff
    
    result = pd.DataFrame({'z-score': z_vals})
    result['p-val'] = sc.norm.sf(abs(result['z-score'])) * 2
    
    return result

In [73]:
df1=tabf1_1.copy()
df2=tabf2_1.copy()
zGDP1=calculate_beta_Z(df1,df2)
#print(zGDP1)

In [74]:
df1=tabf1_2.copy()
df2=tabf2_2.copy()
zGDP2=calculate_beta_Z(df1,df2)
#print(zGDP2)

In [75]:
#print('Z-Values for Coefficients for Agriculture Sector')
df1=Amod_sq_f1.summary_df.copy()
df2=Amod_sq_f2.summary_df.copy()
zA=calculate_beta_Z(df1,df2)
#print(zA)

Z-Values for Coefficients for Agriculture Sector


In [76]:
#print('Z-Values for Coefficients for Manufacturing, Mining and Utilities Sector')
df1=BEmod_sq_f1.summary_df.copy()
df2=BEmod_sq_f2.summary_df.copy()
zBE=calculate_beta_Z(df1,df2)
#print(zBE)

Z-Values for Coefficients for Manufacturing, Mining and Utilities Sector


In [78]:
#print('Z-Values for Coefficients for Transport, Logistics and Storage Sector')
df1=GImod_sq_f1.summary_df.copy()
df2=GImod_sq_f2.summary_df.copy()
zGI=calculate_beta_Z(df1,df2)
#print(zGI)

Z-Values for Coefficients for Transport, Logistics and Storage Sector


In [83]:
zTab=[zGDP1,zGDP2,zA,zBE,zGI]
var_dict = {
        'sq05_colddays': 'Annual Cold Days',
        'sq95_hotdays': 'Annual Hot Days',
        'sq05_dryweeks': 'Annual Dry Weeks',
        'sq95_wetweeks': 'Annual Wet Weeks',
        'sq_std_tas': 'Annual Var. Temp ($^\\circ$C)',
        'sq_std_precpd': 'Annual Var. Precip (mm/day)',
        'sq_mean_precpd': 'Annual Mean Precip (mm/day)',
        'prec': 'Total Annual Precip',
        'tas': 'Mean Annual Temp ($^\\circ$C)',
        'const': 'Constant'
    }

In [90]:
print('\\begin{table}[ht]')
print('  \\centering')
print('  \\caption{Estimation Results: Impact of Weather Anomalies on Growth}')
print('  \\label{tab:regression_results}')

# Column setup: Left align first column, center the rest
col_config = 'l' + 'c' * len(zTab)
print('  \\begin{tabular}{' + col_config + '}')
print('    \\toprule')

# Header Row
header = 'Variable & GDP & GDP$_2$ & A & BE & GI'
print(f'    {header} \\\\')
print('    \\midrule')
for var in var_dict:
    line_coefs=''
    for tab in zTab:
        val,p=tab.loc[var,'z-score'],tab.loc[var,'p-val']
        stars = ''
        if p <= 0.01: stars = '$^{***}$'
        elif p <= 0.05: stars = '$^{**}$'
        elif p <= 0.1: stars = '$^{*}$'

        # Format to 4 decimal places
        line_coefs += f' & {val:.4f}{stars}'
    print(var_dict[var],line_coefs)
print('    \\bottomrule')
print('    \\multicolumn{' + str(len(zTab)+1) + '}{l}{\\small \\textit{Notes:} Standard errors in parentheses. $^{***}$ p$<$0.01, $^{**}$ p$<$0.05, $^{*}$ p$<$0.1.} \\\\')
print('    \\multicolumn{' + str(len(zTab)+1) + '}{l}{\\small \\textit{Notes:} Indicates difference in coefficients of models (3) and (4) in table \ref{tab:lngdppc_results}} \\\\')
print('    \\multicolumn{' + str(len(zTab)+1) + '}{l}{\small \textit{Sectoral Breakdown: }(A) Agriculture, Forestry, Fishery; (B–E) Manufacturing, Mining, Utilities; (G–I) Transport, Logistics, Storage.} \\\\') 
print('  \\end{tabular}')
print('\\end{table}')

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lccccc}
    \toprule
    Variable & GDP & GDP$_2$ & A & BE & GI \\
    \midrule
Annual Cold Days  & 0.0447 & -0.2121 & -1.8486$^{*}$ & -0.6010 & 0.6718
Annual Hot Days  & 0.0000 & -0.0707 & -1.0253 & 0.6835 & 0.0707
Annual Dry Weeks  & 0.0707 & 0.0236 & -0.3969 & 0.3182 & -0.5893
Annual Wet Weeks  & -0.4243 & -0.2828 & -0.4732 & -0.1635 & 0.2121
Annual Var. Temp ($^\circ$C)  & 0.3716 & 0.3961 & 1.4186 & -0.1510 & -0.3880
Annual Var. Precip (mm/day)  & 0.0064 & 0.0778 & 0.0447 & -0.0052 & -0.0193
Annual Mean Precip (mm/day)  & 0.0372 & 0.0884 & 0.0514 & -0.0123 & 0.1026
Total Annual Precip  & 0.0707 & 0.0000 & 0.0354 & 0.0943 & -0.1414
Mean Annual Temp ($^\circ$C)  & 0.4396 & 0.4530 & -0.0387 & -0.5803 & 0.3191
Constant  & -0.4786 & -0.4636 & -0.4076 & 0.5606 & -0.1639
    \bottomrule
    \multicolumn{6}{l}{\small \textit{Notes:} Standar

## Per-Capita GDP with 2 Lags

In [8]:
#%% gdp-percapita growth rate (f1-history), two-fe and quadratic county time trends and two lags
gdppc_f1=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=True,lag_n=2,lag_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False)
gdppc_f1._create_county_trends()
gdppc_f1._create_fixed_effect_dummies()
gdppc_f1._projection_matrix()
gdppc_f1._do_OLS()
gdppc_f1._do_anova()
print("Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.")
gdppc_f1._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
Making 2 Lags: ...
First year = 1992
Last year = 2019
self.regdf.shape =  (10000, 30)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 28
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8752
Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent va

In [9]:
tabf1_1_lags=gdppc_f1.summary_df ##adj_r2=0.23, within_r2=0.01

In [10]:
#%% gdp-percapita growth rate (f2-history), two-fe and quadratic county time trends
gdppc_f2=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f2',
                                                    'sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=True,lag_n=2,lag_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False)
gdppc_f2._create_county_trends()
gdppc_f2._create_fixed_effect_dummies()
gdppc_f2._projection_matrix()
gdppc_f2._do_OLS()
gdppc_f2._do_anova()
print("Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.")
gdppc_f2._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
Making 2 Lags: ...
First year = 1992
Last year = 2019
self.regdf.shape =  (10000, 30)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 28
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8752
Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent v

In [11]:
tabf2_1_lags=gdppc_f2.summary_df

In [12]:
#%% gdp-percapita growth rate (f1-history), two-fe and no county time trend, clustering standard errors for counties

gdppc_f1=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=True,lag_n=2,lag_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False)
#gdppc_f1._create_county_trends(add_cube=False, add_quad=False)
gdppc_f1._create_fixed_effect_dummies(absorb_fe='both')
gdppc_f1._projection_matrix()
gdppc_f1._do_OLS()
gdppc_f1._do_anova()
print("Model with Seasonal Adjustments and Fixed Historical Reference Period.")
gdppc_f1._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
Making 2 Lags: ...
First year = 1992
Last year = 2019
self.regdf.shape =  (10000, 30)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 28
The number of trends = 0
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 423)
Shape of X Matrix = (10000, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 9550
Model with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
------------------------------------------

In [13]:
tabf1_2_lags=gdppc_f1.summary_df

In [14]:
#%% gdp-percapita growth rate (f2-history), two-fe and no county time trend, clustering standard errors for counties

gdppc_f2=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=True,lag_n=2,lag_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False)
#gdppc_f2._create_county_trends(add_cube=False, add_quad=False)
gdppc_f2._create_fixed_effect_dummies(absorb_fe='both')
gdppc_f2._projection_matrix()
gdppc_f2._do_OLS()
gdppc_f2._do_anova()
print("Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.")
gdppc_f2._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
Making 2 Lags: ...
First year = 1992
Last year = 2019
self.regdf.shape =  (10000, 30)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 28
The number of trends = 0
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 423)
Shape of X Matrix = (10000, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 9550
Model with Seasonal Adjustments and Moving Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
--------------------

In [15]:
tabf2_2_lags=gdppc_f2.summary_df

In [21]:
design_table(resdfs=[np.round(tabf1_1_lags,4),np.round(tabf2_1_lags,4),
                     np.round(tabf1_2_lags,4),np.round(tabf2_2_lags,4)],time_trends=['County Quadratic','County Quadratic',
                                                                   'None','None'],
             dummies=['county and year','county and year','county and year','county and year'],
             years=['1995-2019','1995-2019','1995-2019','1995-2019'],counties=['400','400','400','400'],
             whn_r2=['0.00','0.00','0.00','0.00'],
             adj_r2=['0.23','0.23','0.24','0.24'],
             history=['Fixed','Moving','Fixed','Moving'],are_lags=True)

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lcccc}
    \toprule
    Variable & (1) & (2) & (3) & (4) \\
    \midrule
    Annual Cold Days & -0.0004$^{***}$ & -0.0005$^{***}$ & -0.0004$^{***}$ & -0.0004$^{***}$ \\
     & (0.0002) & (0.0002) & (0.0002) & (0.0001) \\
    \addlinespace
    Annual Hot Days & -0.0003$^{***}$ & -0.0004$^{***}$ & -0.0004$^{***}$ & -0.0004$^{***}$ \\
     & (0.0001) & (0.0001) & (0.0001) & (0.0001) \\
    \addlinespace
    Annual Dry Weeks & -0.0002 & -0.0004 & -0.0004 & -0.0005$^{*}$ \\
     & (0.0003) & (0.0003) & (0.0003) & (0.0003) \\
    \addlinespace
    Annual Wet Weeks & -0.0002 & 0.0001 & -0.0004 & -0.0003 \\
     & (0.0003) & (0.0003) & (0.0003) & (0.0003) \\
    \addlinespace
    Annual Var. Temp ($^\circ$C) & 0.0113$^{***}$ & 0.0123$^{***}$ & 0.0143$^{***}$ & 0.0138$^{***}$ \\
     & (0.0039) & (0.0032) & (0.0034) & (0.0029) \\
    \addlinespa

## Sectoral Growth with 2 Lags

# Sector A 

In [22]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F1_ALL
Amod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],
                            make_lags=True,lag_n=2,
                            lag_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False)
Amod_sq_f1._create_county_trends()
Amod_sq_f1._create_fixed_effect_dummies()
Amod_sq_f1._projection_matrix()
Amod_sq_f1._do_OLS()
Amod_sq_f1._do_anova()
print("Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.")
Amod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
Making 2 Lags: ...
First year = 1998
Last year = 2019
self.regdf.shape =  (8800, 30)
Time Frame = 1998 - 2019
Number of years = 22
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 420 dummy columns.
The years of observations = 1998-2019
The total number of counties = 400
The total number of obervations = 8800
The number of regression variables = 28
The number of trends = 798
The number of dummies = 420
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (8800, 1218)
Shape of X Matrix = (8800, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 7555
Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_

In [23]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F2_ALL
Amod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=True,lag_n=2,
                            lag_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'])
Amod_sq_f2._create_county_trends()
Amod_sq_f2._create_fixed_effect_dummies()
Amod_sq_f2._projection_matrix()
Amod_sq_f2._do_OLS()
Amod_sq_f2._do_anova()
print("Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.")
Amod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
Making 2 Lags: ...
First year = 1998
Last year = 2019
self.regdf.shape =  (8800, 30)
Time Frame = 1998 - 2019
Number of years = 22
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 420 dummy columns.
The years of observations = 1998-2019
The total number of counties = 400
The total number of obervations = 8800
The number of regression variables = 28
The number of trends = 798
The number of dummies = 420
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (8800, 1218)
Shape of X Matrix = (8800, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 7555
Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = l

# Sector BE

In [24]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_F1_ALL
BEmod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=True,lag_n=2,
                            lag_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'])
BEmod_sq_f1._create_county_trends()
BEmod_sq_f1._create_fixed_effect_dummies()
BEmod_sq_f1._projection_matrix()
BEmod_sq_f1._do_OLS()
BEmod_sq_f1._do_anova()
print("Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.")
BEmod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
Making 2 Lags: ...
First year = 1998
Last year = 2019
self.regdf.shape =  (8800, 30)
Time Frame = 1998 - 2019
Number of years = 22
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 420 dummy columns.
The years of observations = 1998-2019
The total number of counties = 400
The total number of obervations = 8800
The number of regression variables = 28
The number of trends = 798
The number of dummies = 420
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (8800, 1218)
Shape of X Matrix = (8800, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 7555
Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln

In [25]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_F2_ALL
BEmod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                           make_lags=True,lag_n=2,
                            lag_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'])
BEmod_sq_f2._create_county_trends()
BEmod_sq_f2._create_fixed_effect_dummies()
BEmod_sq_f2._projection_matrix()
BEmod_sq_f2._do_OLS()
BEmod_sq_f2._do_anova()
print("Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.")
BEmod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
Making 2 Lags: ...
First year = 1998
Last year = 2019
self.regdf.shape =  (8800, 30)
Time Frame = 1998 - 2019
Number of years = 22
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 420 dummy columns.
The years of observations = 1998-2019
The total number of counties = 400
The total number of obervations = 8800
The number of regression variables = 28
The number of trends = 798
The number of dummies = 420
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (8800, 1218)
Shape of X Matrix = (8800, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 7555
Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = l

# Sector GI

In [26]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_F1_ALL
GImod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=True,lag_n=2,
                            lag_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'])
GImod_sq_f1._create_county_trends()
GImod_sq_f1._create_fixed_effect_dummies()
GImod_sq_f1._projection_matrix()
GImod_sq_f1._do_OLS()
GImod_sq_f1._do_anova()
print("Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.")
GImod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
Making 2 Lags: ...
First year = 1998
Last year = 2019
self.regdf.shape =  (8800, 30)
Time Frame = 1998 - 2019
Number of years = 22
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 420 dummy columns.
The years of observations = 1998-2019
The total number of counties = 400
The total number of obervations = 8800
The number of regression variables = 28
The number of trends = 798
The number of dummies = 420
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (8800, 1218)
Shape of X Matrix = (8800, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 7555
Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln

In [28]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_F2_ALL
GImod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=True,lag_n=2,
                            lag_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'])
GImod_sq_f2._create_county_trends()
GImod_sq_f2._create_fixed_effect_dummies()
GImod_sq_f2._projection_matrix()
GImod_sq_f2._do_OLS()
GImod_sq_f2._do_anova()
print("Model for Sector GI with Seasonal Adjustments and Rolling Historical Reference Period.")
GImod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
Making 2 Lags: ...
First year = 1998
Last year = 2019
self.regdf.shape =  (8800, 30)
Time Frame = 1998 - 2019
Number of years = 22
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 420 dummy columns.
The years of observations = 1998-2019
The total number of counties = 400
The total number of obervations = 8800
The number of regression variables = 28
The number of trends = 798
The number of dummies = 420
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (8800, 1218)
Shape of X Matrix = (8800, 28)
Estimated 28 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 7555
Model for Sector GI with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = 

In [30]:
design_table(resdfs=[Amod_sq_f1.summary_df,Amod_sq_f2.summary_df,
                     BEmod_sq_f1.summary_df,BEmod_sq_f2.summary_df,
                     GImod_sq_f1.summary_df,GImod_sq_f2.summary_df],
             time_trends=['Quad County']*6,
             dummies=['county and year']*6,
             years=['1996-2019']*6,
             counties=['400']*6,
             whn_r2=['0']*6,
             adj_r2=['0']*6,
             history=['Fixed','Moving']*3,are_lags=True)

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lcccccc}
    \toprule
    Variable & (1) & (2) & (3) & (4) & (5) & (6) \\
    \midrule
    Annual Cold Days & -0.0031$^{***}$ & -0.0008 & -0.0000 & -0.0001 & -0.0005$^{**}$ & -0.0006$^{***}$ \\
     & (0.0008) & (0.0009) & (0.0005) & (0.0004) & (0.0002) & (0.0002) \\
    \addlinespace
    Annual Hot Days & -0.0003 & 0.0004 & 0.0001 & -0.0004 & -0.0002 & -0.0001 \\
     & (0.0005) & (0.0004) & (0.0003) & (0.0003) & (0.0001) & (0.0001) \\
    \addlinespace
    Annual Dry Weeks & 0.0003 & 0.0011 & 0.0000 & -0.0006 & -0.0006$^{*}$ & -0.0003 \\
     & (0.0011) & (0.0012) & (0.0009) & (0.0008) & (0.0003) & (0.0003) \\
    \addlinespace
    Annual Wet Weeks & -0.0006 & 0.0003 & -0.0022$^{**}$ & -0.0014 & 0.0007$^{*}$ & 0.0004 \\
     & (0.0014) & (0.0014) & (0.0010) & (0.0010) & (0.0004) & (0.0004) \\
    \addlinespace
    Annual Var. Temp ($^

# Interaction Terms in the Panel Regression

In [24]:
## adding interaction terms (1) -- tas X std_tas and tas X Extremes

gdppcx_f1=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f1','tasXcold05_f1',
                                                     'tasXdry05_f1','tasXwet95_f1'
                                                    ],make_lags=False,make_vars=True)
gdppcx_f1._create_county_trends()
gdppcx_f1._create_fixed_effect_dummies()
gdppcx_f1._projection_matrix()
gdppcx_f1._do_OLS()
gdppcx_f1._do_anova()
print("Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.")
gdppcx_f1._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 17)
self.regdf.shape =  (10000, 17)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 15
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8765
Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
---------------------------------

In [25]:
## adding interaction terms (2) -- tas X std_tas and tas X Extremes

gdppcx_f2=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f2','tasXcold05_f2',
                                                     'tasXdry05_f2','tasXwet95_f2'
                                                    ],make_lags=False,make_vars=True)
gdppcx_f2._create_county_trends()
gdppcx_f2._create_fixed_effect_dummies()
gdppcx_f2._projection_matrix()
gdppcx_f2._do_OLS()
gdppcx_f2._do_anova()
print("Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.")
gdppcx_f2._model_summary()


Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 17)
self.regdf.shape =  (10000, 17)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 15
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8765
Model with Seasonal Adjustments and Fixed Historical Reference Period, and with added lags.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
---------------------------------

In [38]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F1_ALL
Amod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f1','tasXcold05_f1',
                                                     'tasXdry05_f1','tasXwet95_f1'
                                                    ],make_vars=True,
                            make_lags=False)
Amod_sq_f1._create_county_trends()
Amod_sq_f1._create_fixed_effect_dummies()
Amod_sq_f1._projection_matrix()
Amod_sq_f1._do_OLS()
Amod_sq_f1._do_anova()
print("Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.")
Amod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 17)
self.regdf.shape =  (9600, 17)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 15
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8366
Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_A.d1
-------------------------------------------

In [27]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_f2_ALL
Amod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f2','tasXcold05_f2',
                                                     'tasXdry05_f2','tasXwet95_f2'
                                                    ],make_vars=True,
                            make_lags=False)
Amod_sq_f2._create_county_trends()
Amod_sq_f2._create_fixed_effect_dummies()
Amod_sq_f2._projection_matrix()
Amod_sq_f2._do_OLS()
Amod_sq_f2._do_anova()
print("Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.")
Amod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 17)
self.regdf.shape =  (9600, 17)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 15
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8366
Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_A.d1
-------------------------------------------

In [28]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_f1_ALL
BEmod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f1','tasXcold05_f1',
                                                     'tasXdry05_f1','tasXwet95_f1'
                                                    ],make_vars=True,
                            make_lags=False)
BEmod_sq_f1._create_county_trends()
BEmod_sq_f1._create_fixed_effect_dummies()
BEmod_sq_f1._projection_matrix()
BEmod_sq_f1._do_OLS()
BEmod_sq_f1._do_anova()
print("Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.")
BEmod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 17)
self.regdf.shape =  (9600, 17)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 15
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8366
Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_BE.d1
-----------------------------------------

In [29]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_f2_ALL
BEmod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f2','tasXcold05_f2',
                                                     'tasXdry05_f2','tasXwet95_f2'
                                                    ],make_vars=True,
                            make_lags=False)
BEmod_sq_f2._create_county_trends()
BEmod_sq_f2._create_fixed_effect_dummies()
BEmod_sq_f2._projection_matrix()
BEmod_sq_f2._do_OLS()
BEmod_sq_f2._do_anova()
print("Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.")
BEmod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 17)
self.regdf.shape =  (9600, 17)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 15
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8366
Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_BE.d1
-----------------------------------------

In [30]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_f1_ALL
GImod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f1','tasXcold05_f1',
                                                     'tasXdry05_f1','tasXwet95_f1'
                                                    ],make_vars=True,
                            make_lags=False)
GImod_sq_f1._create_county_trends()
GImod_sq_f1._create_fixed_effect_dummies()
GImod_sq_f1._projection_matrix()
GImod_sq_f1._do_OLS()
GImod_sq_f1._do_anova()
print("Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.")
GImod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 17)
self.regdf.shape =  (9600, 17)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 15
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8366
Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_GI.d1
-----------------------------------------

In [31]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_f2_ALL
GImod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec','tasXsq_std_tas','tasXhot95_f2','tasXcold05_f2',
                                                     'tasXdry05_f2','tasXwet95_f2'
                                                    ],make_vars=True,
                            make_lags=False)
GImod_sq_f2._create_county_trends()
GImod_sq_f2._create_fixed_effect_dummies()
GImod_sq_f2._projection_matrix()
GImod_sq_f2._do_OLS()
GImod_sq_f2._do_anova()
print("Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.")
GImod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 17)
self.regdf.shape =  (9600, 17)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 15
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 15)
Estimated 15 Coefficients from Regression.
Standard Errors are clustered at the county-level
The degrees of freedom = 8366
Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_GI.d1
-----------------------------------------

In [51]:
tab1,tab2=gdppcx_f1.summary_df,gdppcx_f2.summary_df
tab3,tab4=Amod_sq_f1.summary_df,Amod_sq_f2.summary_df
tab5,tab6=BEmod_sq_f1.summary_df,BEmod_sq_f2.summary_df
tab7,tab8=GImod_sq_f1.summary_df,GImod_sq_f2.summary_df
r1,r2=np.round(gdppcx_f1.adjusted_r2,2),np.round(gdppcx_f2.adjusted_r2,2)
r3,r4=np.round(Amod_sq_f1.adjusted_r2,2),np.round(Amod_sq_f2.adjusted_r2,2)
r5,r6=np.round(BEmod_sq_f1.adjusted_r2,2),np.round(BEmod_sq_f2.adjusted_r2,2)
r7,r8=np.round(GImod_sq_f1.adjusted_r2,2),np.round(GImod_sq_f2.adjusted_r2,2)
wr1,wr2=np.round(gdppcx_f1.within_r2,2),np.round(gdppcx_f2.within_r2,2)
wr3,wr4=np.round(Amod_sq_f1.within_r2,2),np.round(Amod_sq_f2.within_r2,2)
wr5,wr6=np.round(BEmod_sq_f1.within_r2,2),np.round(BEmod_sq_f2.within_r2,2)
wr7,wr8=np.round(GImod_sq_f1.within_r2,2),np.round(GImod_sq_f2.within_r2,2)

In [52]:
design_table(resdfs=[np.round(tab1,4),np.round(tab2,4),np.round(tab3,4),np.round(tab4,4),
                     np.round(tab5,4),np.round(tab6,4),np.round(tab7,4),np.round(tab8,4)],
             time_trends=['County Quadratic','County Quadratic',
                          'County Quadratic','County Quadratic',
                          'County Quadratic','County Quadratic',
                          'County Quadratic','County Quadratic'],
             dummies=['county and year','county and year','county and year','county and year',
                     'county and year','county and year','county and year','county and year'],
             years=['1995-2019','1995-2019','1996-2019','1996-2019','1996-2019','1996-2019','1996-2019','1996-2019'],
             counties=['400','400','400','400','400','400','400','400'],
             whn_r2=[wr1,wr2,wr3,wr4,wr5,wr6,wr7,wr8],
             adj_r2=[r1,r2,r3,r3,r5,r6,r7,r8],
             history=['Fixed','Moving','Fixed','Moving','Fixed','Moving','Fixed','Moving'],
             are_interact=True)

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lcccccccc}
    \toprule
    Variable & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) \\
    \midrule
    Annual Cold Days & 0.0006 & -0.0000 & 0.0055$^{*}$ & 0.0045 & -0.0025 & -0.0025 & -0.0010 & -0.0013$^{*}$ \\
     & (0.0007) & (0.0007) & (0.0030) & (0.0030) & (0.0019) & (0.0018) & (0.0008) & (0.0008) \\
    \addlinespace
    Annual Hot Days & -0.0013$^{***}$ & -0.0016$^{***}$ & 0.0052$^{***}$ & 0.0047$^{***}$ & -0.0027$^{***}$ & -0.0030$^{**}$ & 0.0009$^{*}$ & 0.0011$^{**}$ \\
     & (0.0004) & (0.0005) & (0.0015) & (0.0018) & (0.0011) & (0.0013) & (0.0005) & (0.0006) \\
    \addlinespace
    Annual Dry Weeks & -0.0002 & -0.0005 & 0.0010 & 0.0036 & 0.0047 & 0.0041 & 0.0003 & -0.0011 \\
     & (0.0017) & (0.0017) & (0.0063) & (0.0064) & (0.0040) & (0.0039) & (0.0020) & (0.0020) \\
    \addlinespace
    Annual Wet Weeks & -0.0013 & 0

## Conley Standard Errors

In [119]:
## making the conley weights
import geopandas as gpd
shp_kreise=gpd.read_file("C://Users/nb/Desktop/paper_one/official_de_shapefiles.gk3.shape.ebenen/vg5000_ebenen_1231/VG5000_KRS.shp",crs=4326)
shp_kreise=shp_kreise[['ARS','NUTS','geometry']]
shp_kreise['centroid']=shp_kreise['geometry'].centroid
shp_kreise['centroid_deg']=shp_kreise.centroid.to_crs(epsg=4326)
shp_kreise['lat_deg']=shp_kreise.centroid_deg.y
shp_kreise['lon_deg']=shp_kreise.centroid_deg.x
distances=shp_kreise[['ARS','lon_deg','lat_deg']]
distances=distances.rename(columns={'ARS':'Kreise_Code'})
distances['Kreise_Code']=distances['Kreise_Code'].astype(int)
x1_rows=distances['lon_deg'].values.reshape(1,-1)
x2_rows=distances['lon_deg'].values.reshape(-1,1)
x2_minus_x1=x2_rows-x1_rows
y1_rows=distances['lat_deg'].values.reshape(1,-1)
y2_rows=distances['lat_deg'].values.reshape(-1,1)
y2_minus_y1=y2_rows-y1_rows
euc_deg=np.sqrt((x2_minus_x1**2)+(y2_minus_y1**2))
euc_dist=euc_deg*111

bound=250
euc_wts=np.maximum(1 - euc_dist / bound, 0)
euc_wts=pd.DataFrame(euc_wts.copy())
kreise_list=list(distances.Kreise_Code.astype(int))
euc_wts.index=kreise_list
euc_wts.columns=kreise_list
#euc_wts.to_csv("C:\\Users/nb/Desktop/paper_one/conley_weights.csv")

1. there are N total observations {kreise,year}$_{i=1}^N$ and N=10000



2. each $i$-th observation gets multiplied by the inner matrix of {kreise,year}$_{j=1}^N$, 
   and this inner product is weighted by the $\omega_{i,j}$ Conley weight
   
   

3. product matrix is the $\tilde{X}_{i=1,k=1}^{N,K} \cdot \tilde{u}_{i=1}^N$ matrix

In [15]:
gdppc_f1=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=False,make_vars=False)
gdppc_f1._create_county_trends()
gdppc_f1._create_fixed_effect_dummies()
gdppc_f1._projection_matrix()
gdppc_f1._do_OLS()
gdppc_f1._do_anova(cluster='conley')
print("Model with Seasonal Adjustments and Fixed Historical Reference Period, Conley SE.")
gdppc_f1._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
self.regdf.shape =  (10000, 12)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 10
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8770
Model with Seasonal Adjustments and Fixed Historical Reference Period, Conley SE.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
------------------------------

In [16]:
gdppc_f2=panel_regression(years="1995-2019",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_lags=False,make_vars=False)
gdppc_f2._create_county_trends()
gdppc_f2._create_fixed_effect_dummies()
gdppc_f2._projection_matrix()
gdppc_f2._do_OLS()
gdppc_f2._do_anova(cluster='conley')
print("Model with Seasonal Adjustments and Fixed Historical Reference Period, Conley SE.")
gdppc_f2._model_summary()

Shape of original dataframe = (12400, 39)
Continuing with fulldf
Shape of the dataframe = (12400, 12)
self.regdf.shape =  (10000, 12)
Time Frame = 1995 - 2019
Number of years = 25
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 423 dummy columns.
The years of observations = 1995-2019
The total number of counties = 400
The total number of obervations = 10000
The number of regression variables = 10
The number of trends = 798
The number of dummies = 423
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (10000, 1221)
Shape of X Matrix = (10000, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8770
Model with Seasonal Adjustments and Fixed Historical Reference Period, Conley SE.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gdppc.d1
------------------------------

# Sector A 

In [17]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F1_ALL
Amod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
Amod_sq_f1._create_county_trends()
Amod_sq_f1._create_fixed_effect_dummies()
Amod_sq_f1._projection_matrix()
Amod_sq_f1._do_OLS()
Amod_sq_f1._do_anova(cluster='conley')
print("Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.")
Amod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8371
Model for Sector A with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_A.d1
------------------------------

In [18]:
#%% sector-A productivity growth without lags, model code GVAPWc_A_F2_ALL
Amod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_A.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
Amod_sq_f2._create_county_trends()
Amod_sq_f2._create_fixed_effect_dummies()
Amod_sq_f2._projection_matrix()
Amod_sq_f2._do_OLS()
Amod_sq_f2._do_anova(cluster='conley')
print("Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.")
Amod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8371
Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_A.d1
----------------------------

# Sector BE

In [19]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_F1_ALL
BEmod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
BEmod_sq_f1._create_county_trends()
BEmod_sq_f1._create_fixed_effect_dummies()
BEmod_sq_f1._projection_matrix()
BEmod_sq_f1._do_OLS()
BEmod_sq_f1._do_anova(cluster='conley')
print("Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.")
BEmod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8371
Model for Sector BE with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_BE.d1
----------------------------

In [20]:
#%% sector-BE productivity growth without lags, model code GVAPWc_BE_F2_ALL
BEmod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_BE.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
BEmod_sq_f2._create_county_trends()
BEmod_sq_f2._create_fixed_effect_dummies()
BEmod_sq_f2._projection_matrix()
BEmod_sq_f2._do_OLS()
BEmod_sq_f2._do_anova(cluster='conley')
print("Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.")
BEmod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8371
Model for Sector A with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_BE.d1
---------------------------

# Sector GI

In [21]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_F1_ALL
GImod_sq_f1=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f1','sq95_hotdays_f1',
                                     'sq05_dryweeks_f1','sq95_wetweeks_f1',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
GImod_sq_f1._create_county_trends()
GImod_sq_f1._create_fixed_effect_dummies()
GImod_sq_f1._projection_matrix()
GImod_sq_f1._do_OLS()
GImod_sq_f1._do_anova(cluster='conley')
print("Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.")
GImod_sq_f1._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8371
Model for Sector GI with Seasonal Adjustments and Fixed Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_GI.d1
----------------------------

In [22]:
#%% sector-GI productivity growth without lags, model code GVAPWc_GI_F2_ALL
GImod_sq_f2=panel_regression(filename="all_sectors_ardeco_c8.csv",years="1995-2019",
                            y_var="ln_gvapw_GI.d1",x_vars=['sq05_colddays_f2','sq95_hotdays_f2',
                                     'sq05_dryweeks_f2','sq95_wetweeks_f2',
                                     'sq_mean_precpd','sq_std_precpd','sq_std_tas',
                                     'tas','prec'],make_vars=False,
                            make_lags=False)
GImod_sq_f2._create_county_trends()
GImod_sq_f2._create_fixed_effect_dummies()
GImod_sq_f2._projection_matrix()
GImod_sq_f2._do_OLS()
GImod_sq_f2._do_anova(cluster='conley')
print("Model for Sector GI with Seasonal Adjustments and Rolling Historical Reference Period.")
GImod_sq_f2._model_summary()

Shape of original dataframe = (10400, 48)
Continuing with fulldf
Shape of the dataframe = (10400, 12)
self.regdf.shape =  (9600, 12)
Time Frame = 1996 - 2019
Number of years = 24
Number of counties = 400
Is the panel balanced ? =  True
Created 798 county time trend columns.
Created 422 dummy columns.
The years of observations = 1996-2019
The total number of counties = 400
The total number of obervations = 9600
The number of regression variables = 10
The number of trends = 798
The number of dummies = 422
Checking if the D matrix is full rank ...
D matrix is at full rank. Proceed. uWu.
Shape of D Matrix = (9600, 1220)
Shape of X Matrix = (9600, 10)
Estimated 10 Coefficients from Regression.
Standard Errors are Conley Standard Errors with bound of 250km
The degrees of freedom = 8371
Model for Sector GI with Seasonal Adjustments and Rolling Historical Reference Period.
uWu ************* RESULTS SUMMARY ***************** uWu
The dependent variable = ln_gvapw_GI.d1
--------------------------

In [23]:
design_table(resdfs=[gdppc_f1.summary_df,gdppc_f2.summary_df,
                     Amod_sq_f1.summary_df,Amod_sq_f2.summary_df,
                     BEmod_sq_f1.summary_df,BEmod_sq_f2.summary_df,
                     GImod_sq_f1.summary_df,GImod_sq_f2.summary_df],
             time_trends=['Quad County']*8,
             dummies=['county and year']*8,
             years=['1995-2019']*2+['1996-2019']*6,
             counties=['400']*8,
             whn_r2=['0']*8,
             adj_r2=['0']*8,
             history=['Fixed','Moving']*4)

\begin{table}[ht]
  \centering
  \caption{Estimation Results: Impact of Weather Anomalies on Growth}
  \label{tab:regression_results}
  \begin{tabular}{lcccccccc}
    \toprule
    Variable & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) \\
    \midrule
    Annual Cold Days & -0.0004$^{***}$ & -0.0004$^{***}$ & -0.0034$^{**}$ & -0.0015 & -0.0005 & -0.0002 & -0.0002 & -0.0004 \\
     & (0.0001) & (0.0001) & (0.0016) & (0.0016) & (0.0004) & (0.0003) & (0.0004) & (0.0004) \\
    \addlinespace
    Annual Hot Days & -0.0004$^{***}$ & -0.0004$^{***}$ & -0.0006 & -0.0000 & -0.0000 & -0.0003 & 0.0000 & -0.0000 \\
     & (0.0001) & (0.0001) & (0.0007) & (0.0007) & (0.0003) & (0.0003) & (0.0002) & (0.0002) \\
    \addlinespace
    Annual Dry Weeks & -0.0003 & -0.0003 & 0.0001 & 0.0007 & -0.0001 & -0.0005 & -0.0006$^{*}$ & -0.0003 \\
     & (0.0004) & (0.0004) & (0.0013) & (0.0017) & (0.0009) & (0.0010) & (0.0004) & (0.0004) \\
    \addlinespace
    Annual Wet Weeks & -0.0002 & -0.0000 & 0.0011 & 